# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [1]:
from explanation.explanation import *
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain_ollama import OllamaLLM

In [2]:
template_path = "explanation/explanation_template.txt"

In [3]:
# Llama 3.2 3B
llm = OllamaLLM(
    model = 'llama3.2'
)

Without code comments

In [24]:
prolog_input = compose_prolog_input(['linear_path'], comments=False)
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, prolog_input)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

linearpath(A,B):- same_gate(A,B).
linearpath(A,B):- not(branches(A)),is_connected(A,C),linearpath(C,B).

not_linearpath(A,B) :- not(linearpath(A,B)).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

branches(A) :- is_connected(A, B), is_connected(A, C), B \== C.
not_branches(A) :- not(branches(A)).

same_gate(A, A).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%

direction(linearpath,(in,out)). 
direction(branches,(in,)).
direction(not_branches,(in,)).
direction(is_connected,(in,out)).
direction(same_gate,(in,out)).


===== Generated Raw Text =====
{"summary": "This Prolog program determines whether two nodes in a graph have a direct linear path between them and flags non-linear paths.", "primitive": [{"name": "branches", "description": "Checks if there are at least two connected components in the graph with different edges."}, {"name": "is_connected", "description": "Verifies that two nodes in the graph are part of the

In [21]:
partition_sizes = compose_prolog_input(['partition', 'partition_sizes'], comments=False)
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, partition_sizes)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%


inv_ho_0(A,B):- same_circuit(A,B),linearpath(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linearpath(B,A)).

partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

partition_sizes(A,B):- partition(A,E,C),size(C,F),size(E,D),pair(D,F,B).
partition_sizes(A,B):- partition(A,E,C),size(C,F),size(E,D),pair(F,D,B).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

all(P, A, L):- 
    findall(H, call(P, A, H), L).

same_circuit(A, B) :- gate(A), gate(B), N is A // 100, M is B // 100, N == M.
size(L, S) :- is_list(L), length(L, S).

pair(A,B,[A,B]) :- not_list(A), not_list(B).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%

direction(partition,(in,out,out)). 
direction(linearpath,(in,out)).
direction(not_linearpath,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).
direction(partition_sizes,(in,out)). 
direction(partition,(in,out,out)).
direction(size,(in,out)).
direction(pair,(in,in,out)).


===== G

In [22]:
partition_sizes = compose_prolog_input(['select_test'], comments=False)
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, partition_sizes)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%


select_test(A,B):- empty_partitions(D),map(partition_sizes,A,C),fold(max_min_size,D,C,B).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

max_min_size([A,B], [C,D], [A,B]) :- not_list(A), not_list(B), not_list(C), not_list(D), max(A, C, A).
max_min_size([A,B], [C,D], [C,D]) :- not_list(A), not_list(B), not_list(C), not_list(D), max(A, C, C).

min(A, B, C) :- min_list([A,B],C).
max(A, B, C) :- max_list([A,B],C).

not_list(A) :- not(is_list(A)).

fold(_P,Acc,[],Acc).
fold(P,Acc,[H|T],Out) :- 
    call(P,Acc,H,Inter),
    fold(P,Inter,T,Out).

map(_P,[],[]).
map(P,[H|T],[H1|T1]) :- 
    call(P,H,H1),
    map(P,T,T1).

empty_partitions([0, 0]).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%

direction(select_test,(in,out)). 
direction(partition_sizes,(in,out)). 
direction(empty_partitions,(out,)).
direction(max_min_size,(in,in,out)).
direction(map,((in,out),in,out)).
direction(fold,((in,in,out),in,in,out)).



===== Gener

With code comments

In [5]:
prolog_input = compose_prolog_input(['linear_path'])
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, prolog_input)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

% Is gate A to gate B a path without branches?
linearpath(A,B):- same_gate(A,B).
linearpath(A,B):- not(branches(A)),is_connected(A,C),linearpath(C,B).

% The negation
not_linearpath(A,B) :- not(linearpath(A,B)).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

% Are multiple gates connected to gate A?
branches(A) :- is_connected(A, B), is_connected(A, C), B \== C.
not_branches(A) :- not(branches(A)).

% Two gates are the same
same_gate(A, A).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%

direction(linearpath,(in,out)). 
direction(branches,(in,)).
direction(not_branches,(in,)).
direction(is_connected,(in,out)).
direction(same_gate,(in,out)).


===== Generated Raw Text =====
{"summary": "This Prolog program determines whether there is a path between two gates in a circuit that does not branch off into multiple paths.", "primitive": [{"name": "branches", "description": "Checks if more than one gate is connected to a give

In [7]:
partition_sizes = compose_prolog_input(['partition', 'partition_sizes'])
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, partition_sizes)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%


% Gates affected by a test and those not affected
inv_ho_0(A,B):- same_circuit(A,B),linearpath(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linearpath(B,A)).

% Given Gate A find gates on the same linear path as A and gates on other paths
partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

% Given a fault, compute the partition sizes and write them into a pair
partition_sizes(A,B):- partition(A,E,C),size(C,F),size(E,D),pair(D,F,B).
partition_sizes(A,B):- partition(A,E,C),size(C,F),size(E,D),pair(F,D,B).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

% Higher Order primitive
all(P, A, L):- 
    findall(H, call(P, A, H), L).

% Does gate A share a linear path with gate B (B precedes A if A \== B)?
same_circuit(A, B) :- gate(A), gate(B), N is A // 100, M is B // 100, N == M.
% length of a list
size(L, S) :- is_list(L), length(L, S).

% Create a pair of two numbers
pair(A,B,[A,B]) :- not_list(A), not_list(B).

%%%%%%%%%

In [19]:
partition_sizes = compose_prolog_input(['select_test'])
with open(template_path, 'r') as file:
        explanation_template = file.read()
explanation = explain_prolog(llm, explanation_template, partition_sizes)



%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%


select_test(A,B):- empty_partitions(D),map(partition_sizes,A,C),fold(max_min_size,D,C,B).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

% Return the partitions with a larger min partition size
max_min_size([A,B], [C,D], [A,B]) :- not_list(A), not_list(B), not_list(C), not_list(D), max(A, C, A).
max_min_size([A,B], [C,D], [C,D]) :- not_list(A), not_list(B), not_list(C), not_list(D), max(A, C, C).

% Find the smaller and larger number of two numbers
min(A, B, C) :- min_list([A,B],C).
max(A, B, C) :- max_list([A,B],C).

% Not a list
not_list(A) :- not(is_list(A)).

% Higher order primitive
% List manipulation predicates
fold(_P,Acc,[],Acc).
fold(P,Acc,[H|T],Out) :- 
    call(P,Acc,H,Inter),
    fold(P,Inter,T,Out).

% Map a predicate over a list
map(_P,[],[]).
map(P,[H|T],[H1|T1]) :- 
    call(P,H,H1),
    map(P,T,T1).

% The smallest partition base case
empty_partitions([0, 0]).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%